In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check the GPU status
!nvidia-smi

Mon Sep 21 13:42:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   78C    P0             32W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries

In [3]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
    "transformers==4.53.3" \
    "peft==0.17.1" \
    "trl" \
    "accelerate" \
    "bitsandbytes" \
    "wandb"

In [4]:
import os
import math
import random
from datetime import datetime

import numpy as np

import torch
from transformers import (
    AutoModelForMaskedLM, 
    AutoModelForQuestionAnswering,
    DataCollatorForLanguageModeling, 
    DataCollatorWithPadding,
    AutoTokenizer,
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback,
)
from peft import (
    LoraConfig, 
    get_peft_model,
    PeftModel,
)
from datasets import load_dataset, Dataset

# Utilities

In [5]:
def load_train_val_datasets(
    task: str,  # 'wikipedia' | 'squad'
    lang: str,  # wikipedia: 'en' | '<any_lang>', squad: 'en' only
    train_size: int, 
    val_size: int,
) -> tuple[Dataset, Dataset]:
    # Validate that if the task is 'squad', the language must be 'en'
    if task == 'squad':
        assert lang == 'en', "SQuAD is English-only."
    
    # Define dataset configurations for each task
    data_configs = {
        'wikipedia': {
            'data_id': 'wikimedia/wikipedia',
            'data_dir': f'20231101.{lang}',
            'train_split': 'train',
            'val_split': 'train',
        },
        'squad': {
            'data_id': 'rajpurkar/squad',
            'data_dir': None,
            'train_split': 'train',
            'val_split': 'validation',
        },
    }
    
    # Validate that the specified task is supported
    assert task in data_configs, (
        f"Unsupported task: {task}. "
        f"Supported tasks: {list(data_configs.keys())}"
    )

    # Set up Hugging Face dataset configuration
    data_id = data_configs[task]['data_id']
    data_dir = data_configs[task]['data_dir']
    train_split = data_configs[task]['train_split']
    val_split = data_configs[task]['val_split']

    if train_split == val_split:
        # If the train and validation splits are the same, we need to sample from the same dataset stream
        dataset_stream = load_dataset(
            data_id,
            data_dir=data_dir,
            split=train_split,
            streaming=True,
        )

        train_data = []
        val_data = []

        for i, example in enumerate(dataset_stream):
            if i < train_size:
                train_data.append(example)
            elif i < train_size + val_size:
                val_data.append(example)
            else:
                break

    else:
        # If the train and validation splits are different, we can sample from each split separately
        def sample_split(split, size):
            dataset_stream = load_dataset(
                data_id,
                data_dir=data_dir,
                split=split,
                streaming=True,
            )

            data = []
            for i, example in enumerate(dataset_stream):
                if i >= size:
                    break
                data.append(example)
            return data

        train_data = sample_split(train_split, train_size)
        val_data = sample_split(val_split, val_size)

    return (
        Dataset.from_list(train_data),
        Dataset.from_list(val_data),
    )

# Configuration

In [6]:
# Run configuration
TASK = 'wikipedia'  # 'wikipedia' | 'squad'
LANG = 'vi'         # wikipedia: 'en' | '<any_lang>', squad: 'en' only
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Data configuration
MLM_PROB = 0.15
# -------- 2K (80%:20%) --------
# TRAIN_SIZE = 2_000
# VAL_SIZE = 500
# -------- 3K (80%:20%) --------
TRAIN_SIZE = 3_000
VAL_SIZE = 750
# -------- 15K (80%:20%) --------
# TRAIN_SIZE = 15_000
# VAL_SIZE = 3_750

# Model configuration
MODEL_ID = 'FacebookAI/xlm-roberta-base'
MODEL_NAME = 'XLM-R-Base'
# -------- New training --------
RESUME_MODEL_ID = None
RESUME_CKPT_STEP = None
# -------- Resume training --------
# RESUME_MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-15K-s42-LoRA-v260920193856'
# RESUME_CKPT_STEP = 8100

# LoRA configuration
LORA_RANK = 8
LORA_ALPHA = 8
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = [
    # Not 'all-linear', since we exclude 'qa_outputs' for question answering tasks
    'query', 
    'key', 
    'value', 
    # Not all 'dense' layers, since we exclude 'lm_head.dense' for masked language modeling
    'intermediate.dense',
    'output.dense',
]

# Training configuration
MINI_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
# LR = 2e-4
# -------- wikipedia-vi-3K | wikipedia-en-2K --------
LR = 1e-4
WARMUP_STEPS = 50
EVAL_STEPS = 100
SAVE_STEPS = 100
LOGGING_STEPS = 50
# ---
# NUM_EPOCHS = 3
# EARLY_STOPPING_PATIENCE = 3
# ---
NUM_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 5
# -------- squad-en-15K --------
# LR = 1e-4
# WARMUP_STEPS = 100
# EVAL_STEPS = 300
# SAVE_STEPS = 300
# LOGGING_STEPS = 100
# ---
# NUM_EPOCHS = 5
# EARLY_STOPPING_PATIENCE = 3
# ---
# NUM_EPOCHS = 5
# EARLY_STOPPING_PATIENCE = 5
# ---
# NUM_EPOCHS = 10
# EARLY_STOPPING_PATIENCE = 5
# ---
# NUM_EPOCHS = 15
# EARLY_STOPPING_PATIENCE = 10

In [7]:
# Resume training configuration
resume_from_checkpoint = None
if RESUME_MODEL_ID:
    model_name = RESUME_MODEL_ID
    run_name = model_name.split('/')[-1]
    hub_model_id = RESUME_MODEL_ID
    
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=hub_model_id, local_dir=model_name)
    
    if RESUME_CKPT_STEP:
        resume_from_checkpoint = f"{hub_model_id}/checkpoint-{RESUME_CKPT_STEP}"
        # Ensure the checkpoint exists
        assert os.path.exists(resume_from_checkpoint), f"Checkpoint {resume_from_checkpoint} does not exist."
else:
    run_name = f'{MODEL_NAME}-{TASK}-{LANG}-{TRAIN_SIZE/1000:g}K-s{SEED}-LoRA-v{datetime.now().strftime("%y%m%d%H%M%S")}'
    
    # Automatically retrieve the username of the currently logged-in user
    from huggingface_hub import HfApi

    user_info = HfApi().whoami()
    username = user_info["name"]

    hub_model_id = f"{username}/{run_name}"
base_hub_model_id, version = hub_model_id.rsplit('-v', 1)
hub_merged_model_id = f'{base_hub_model_id}-mrg-v{version}'

print("Resume from checkpoint:", resume_from_checkpoint)
print("Model name:", MODEL_NAME)
print("Run name:", run_name)
print("Hub model ID:", hub_model_id)
print("Hub merged model ID:", hub_merged_model_id)

Resume from checkpoint: None
Model name: XLM-R-Base
Run name: XLM-R-Base-wikipedia-vi-3K-s42-LoRA-v260921134220
Hub model ID: alxxtexxr/XLM-R-Base-wikipedia-vi-3K-s42-LoRA-v260921134220
Hub merged model ID: alxxtexxr/XLM-R-Base-wikipedia-vi-3K-s42-LoRA-mrg-v260921134220


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


# Model

In [8]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Determine the appropriate model classes based on the task
if TASK == 'squad':
    model_cls = AutoModelForQuestionAnswering
    task_type = 'QUESTION_ANS'
    data_collator = DataCollatorWithPadding(tokenizer)
    label_names = ['start_positions', 'end_positions']
else:
    model_cls = AutoModelForMaskedLM
    task_type = 'FEATURE_EXTRACTION'
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=MLM_PROB,
    )
    label_names = ['labels']

# Load the model
model = model_cls.from_pretrained(MODEL_ID)

Some weights of the model checkpoint at FacebookAI/xlm-roberta-base were not used when initializing XLMRobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [9]:
# Sanity check
# for n, p in model.named_parameters():
#     print(n, p.requires_grad)

In [10]:
if resume_from_checkpoint:
    # Load the LoRA adapter from the checkpoint and ensure it's in training mode
    model = PeftModel.from_pretrained(model, resume_from_checkpoint)
    model.train()                   # Ensure the resumed model is in training mode
    model.enable_adapter_layers()   # Explicitly unfreeze LoRA weights
else:
    # Set up a LoRA configuration and apply it to the model
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias='none',
        task_type=task_type,
        target_modules=LORA_TARGET_MODULES,
    )
    model: PeftModel = get_peft_model(model, lora_config)

model = model.to(DEVICE)

model.print_trainable_parameters()
print("device:", model.device)

trainable params: 1,327,104 || all params: 279,622,290 || trainable%: 0.4746
device: cuda:0


In [11]:
# Sanity check
# for n, p in model.named_parameters():
#     if p.requires_grad:
#         print(n, p.requires_grad)

# Data

In [12]:
# Load the dataset
train_dataset, val_dataset = load_train_val_datasets(
    task=TASK, 
    lang=LANG,
    train_size=TRAIN_SIZE, 
    val_size=VAL_SIZE,
)

print("Train dataset:")
print(train_dataset)
print()
print("Validation dataset:")
print(val_dataset)

Train dataset:
Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 3000
})

Validation dataset:
Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 750
})


In [13]:
# Preprocess the dataset
WIKI_MAX_LENGTH = 510  # 510 + 2 (BOS + EOS) = 512, matching XLM-R's max_position_embeddings
NUM_CHUNKS_PER_WIKI_ARTICLE = 3

if TASK == 'squad':

    def preprocess_squad(examples):
        tokenized = tokenizer(
            examples['question'],
            examples['context'],
            truncation='only_second', # Only truncate the context
            max_length=384,
            stride=128,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding=False,
        )

        # Each generated window points back to its original SQuAD example.
        sample_mapping = tokenized.pop('overflow_to_sample_mapping') # Global index

        # Save offsets separately because the model does not need them.
        offset_mapping = tokenized.pop('offset_mapping') # Local index

        start_positions = []
        end_positions = []

        # Indices of windows that contain the complete answer.
        keep_indices = []

        for feature_idx, offsets in enumerate(offset_mapping):

            # Which original example produced this window?
            sample_idx = sample_mapping[feature_idx]

            answer = examples['answers'][sample_idx]

            answer_start_char = answer['answer_start'][0]
            answer_text = answer['text'][0]
            answer_end_char = (
                answer_start_char + len(answer_text)
            )

            # sequence_ids tells us which tokens belong to:
            # 0 = question
            # 1 = context
            # None = special tokens
            sequence_ids = tokenized.sequence_ids(feature_idx)

            # Find the first and last context tokens in this window.
            context_token_indices = [
                idx
                for idx, seq_id in enumerate(sequence_ids)
                if seq_id == 1
            ]

            if not context_token_indices:
                continue

            context_start = context_token_indices[0]
            context_end = context_token_indices[-1]

            # Check whether the COMPLETE answer is inside this window.
            #
            # If the answer starts before the first context token,
            # or ends after the last context token, this window
            # does not contain the complete answer.
            if offsets[context_start][0] > answer_start_char:
                continue

            if offsets[context_end][1] < answer_end_char:
                continue

            # Find the token containing the answer start.
            token_start = context_start

            while (
                token_start <= context_end
                and offsets[token_start][1] <= answer_start_char
            ):
                token_start += 1

            # Find the token containing the answer end.
            token_end = context_end

            while (
                token_end >= context_start
                and offsets[token_end][0] >= answer_end_char
            ):
                token_end -= 1

            # Safety check.
            if token_start > context_end or token_end < context_start:
                continue

            # Keep this window.
            keep_indices.append(feature_idx)
            start_positions.append(token_start)
            end_positions.append(token_end)

        # Keep only windows containing the complete answer.
        tokenized = {
            key: [
                values[i]
                for i in keep_indices
            ]
            for key, values in tokenized.items()
        }

        tokenized['start_positions'] = start_positions
        tokenized['end_positions'] = end_positions

        return tokenized


    train_dataset = train_dataset.map(
        preprocess_squad,
        batched=True,
        remove_columns=train_dataset.column_names,
    )

    val_dataset = val_dataset.map(
        preprocess_squad,
        batched=True,
        remove_columns=val_dataset.column_names,
    )

else:

    def tokenize_text(examples, indices):
        rng = random.Random(SEED)

        all_input_ids = []
        all_attention_masks = []
        all_labels = []

        for text, article_idx in zip(examples['text'], indices):

            # Tokenize the entire article without truncation.
            # Special tokens are added after chunking.
            article_tokens = tokenizer(
                text,
                truncation=False,
                add_special_tokens=False,
            )['input_ids']

            # Create non-overlapping 512-token chunks.
            chunks = [
                article_tokens[i:i + WIKI_MAX_LENGTH]
                for i in range(
                    0,
                    len(article_tokens),
                    WIKI_MAX_LENGTH,
                )
            ]

            # Keep only complete 512-token chunks.
            chunks = [
                chunk
                for chunk in chunks
                if len(chunk) == WIKI_MAX_LENGTH
            ]

            # Select up to N chunks randomly.
            num_chunks = min(
                NUM_CHUNKS_PER_WIKI_ARTICLE,
                len(chunks),
            )

            if num_chunks == 0:
                continue

            # Use a deterministic per-article RNG.
            article_rng = random.Random(
                SEED + article_idx
            )

            selected_chunks = article_rng.sample(
                chunks,
                num_chunks,
            )

            for chunk in selected_chunks:

                # Add special tokens in the same way
                # the XLM-R tokenizer normally does.
                input_ids = (
                    [tokenizer.bos_token_id]
                    + chunk
                    + [tokenizer.eos_token_id]
                )
                attention_mask = [1] * len(input_ids)

                all_input_ids.append(input_ids)
                all_attention_masks.append(attention_mask)
                all_labels.append(input_ids.copy())

        return {
            'input_ids': all_input_ids,
            'attention_mask': all_attention_masks,
            'labels': all_labels,
        }

    train_dataset = train_dataset.map(
        tokenize_text,
        batched=True,
        with_indices=True,
        remove_columns=train_dataset.column_names,
    )

    val_dataset = val_dataset.map(
        tokenize_text,
        batched=True,
        with_indices=True,
        remove_columns=val_dataset.column_names,
    )

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (8177 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/750 [00:00<?, ? examples/s]

In [14]:
if TASK == "wikipedia":
    print("Total chunks:", len(train_dataset))

Total chunks: 4902


# Training

In [15]:
# Calculate steps per epoch and max steps
steps_per_epoch = math.ceil(len(train_dataset) / (MINI_BATCH_SIZE * GRAD_ACCUM_STEPS))
max_steps = steps_per_epoch * NUM_EPOCHS
print("Steps per epoch:", steps_per_epoch)
print("Max steps:", max_steps)

Steps per epoch: 307
Max steps: 1535


In [16]:
# Initialize wandb
import wandb
wandb.init(
    project="legamex",
    name=run_name,
)

# Compute metrics for SQuAD (token-level span F1 and Exact Match)
def compute_metrics_qa(eval_pred):
    start_logits, end_logits = eval_pred.predictions
    start_positions, end_positions = eval_pred.label_ids

    # Take argmax to get predicted positions
    pred_starts = np.argmax(start_logits, axis=-1)
    pred_ends = np.argmax(end_logits, axis=-1)

    f1_scores = []
    em_scores = []
    for pred_start, pred_end, true_start, true_end in zip(
        pred_starts, pred_ends, start_positions, end_positions
    ):
        # Ensure valid spans
        if pred_start > pred_end:
            pred_start, pred_end = pred_end, pred_start

        # Exact Match: both start and end must match exactly
        em_scores.append(float(pred_start == true_start and pred_end == true_end))

        predicted_tokens = set(range(pred_start, pred_end + 1))
        true_tokens = set(range(true_start, true_end + 1))

        if len(predicted_tokens) == 0 or len(true_tokens) == 0:
            f1_scores.append(0.0)
            continue

        intersection = predicted_tokens & true_tokens
        precision = len(intersection) / len(predicted_tokens)
        recall = len(intersection) / len(true_tokens)

        if precision + recall == 0:
            f1_scores.append(0.0)
        else:
            f1_scores.append(2 * precision * recall / (precision + recall))

    return {'f1': np.mean(f1_scores), 'exact_match': np.mean(em_scores)}

# Set up a trainer
training_args = TrainingArguments(
    # Training arguments
    seed=SEED,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    per_device_train_batch_size=MINI_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    max_steps = max_steps,
    warmup_steps = WARMUP_STEPS,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    max_grad_norm=1.0,
    weight_decay=0.01,
    
    # Validation arguments
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    
    # Logging arguments
    logging_strategy='steps',
    logging_steps=LOGGING_STEPS,
    # logging_first_step=True,
    report_to=['tensorboard', 'wandb'],
    
    # Saving arguments
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    # save_total_limit=5, # 1 best + 4 recent checkpoints. WARN: It doesn't work
    
    # With load_best_model_at_end=True, your save_strategy will be ignored and default to eval_strategy.
    # So you will find one checkpoint at the end of each epoch.
    # https://discuss.huggingface.co/t/trainer-not-saving-after-save-steps/5464
    load_best_model_at_end=True,
    metric_for_best_model='f1' if TASK == 'squad' else 'eval_loss',
    greater_is_better=True if TASK == 'squad' else False,

    # run_name=run_name,
    output_dir=run_name,
    hub_model_id=hub_model_id,
    push_to_hub=True,
    hub_strategy='all_checkpoints',
    hub_always_push=True,
)
trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics_qa if TASK == 'squad' else None,
    # label_names=['labels'],
    callbacks = [
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            # early_stopping_threshold = 0.001,
        )
    ],
)
trainer.label_names = label_names

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: alimtegar to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


No label_names provided for model class `PeftModelForFeatureExtraction`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [17]:
# Start training
trainer_stats = trainer.train(resume_from_checkpoint=resume_from_checkpoint)

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss,Validation Loss
100,1.546400,1.433762
200,1.554500,1.420871
300,1.521500,1.432521
400,1.515500,1.427641
500,1.523300,1.437689
600,1.522200,1.434048
700,1.514300,1.435044


# Merging

In [18]:
# If the task is SQuAD, upload the merged model and tokenizer
if TASK == 'squad':
    # After the training finishes, merge the LoRA into the base model and save everything
    model = model.eval() # Good practice
    merged_model = model.merge_and_unload()

    # Upload the merged model to Hugging Face
    merged_model.push_to_hub(hub_merged_model_id)
    tokenizer.push_to_hub(hub_merged_model_id)

    print(f"Merged model uploaded to: https://huggingface.co/{hub_merged_model_id}")